# QUMMSA

Este algoritmo consiste en hallar el valor máximo o mínimo utilizando Grover Long y el modelo de Durr Hoyer para encontrar con precisión un resultado.

En el presente cuaderno, veremos la implementación, paso a paso, del algoritmo en cuestión. Procurando conservar los mismos nombres de variables para no desviar la comprensión del paper.

Lo principal, importar los paquetes correspondientes para la implementación en IBM Qiskit en Python 3.12.0

In [1]:
import qiskit
import numpy as np
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt
import qiskit_aer


## The QUMMSA
Problem: Let 𝐷 be an unsorted database with 𝑁 items. The problem is to find the maximum or minimum from 𝐷 .

In [2]:
D_original = [ 23, 26, 60, 7, 9, 12, 15, 18]
N = len(D_original)


> Core idea: Exploiting the Grover-Long algorithm, we can find 𝑀(𝑀≥1) solutions from the unsorted database with 𝑁 items. Here, a random value 𝑑0 is taken as a reference value. If the search algorithm gives a result 𝑑1, which is less than or equals to 𝑑0, it will run successfully. Note that there are 𝑀 results that satisfy the search condition and 𝑑1 is one of them. Then, let 𝑑1 replace 𝑑0 and repeat the above steps until 𝑀=1. Since the 𝑀 solutions are given with equal probability after Grover-Long algorithm, the number of solutions will be reduced by half on average, after one main loop. Therefore, the mathematical expectation of main loops to find the minimum is log2𝑁, in theory.

En este ejemplo didáctico, vemos el algoritmo a un nivel meramente clásico que sirve para visualizar las iteraciones de la búsqueda siguiendo lo estipulado en el paper.

In [25]:

D = D_original
d_0 = np.random.choice(D)
d = [] 
M = N
# M = len(d)
while M > 1:
    d = []
    for i in D:
        if i <= d_0:
            d.append(i)
    M = len(d)
    print(M)
    d_1 = np.random.choice(d)
    d_0 = d_1

3
1


12/07/26
A continuación, intenté recrear el pseudocódigo del paper, sin embargo después de corregir la posición dentro del while de d_1 = 999, sigo sin entender por qué la condición de parada es c y con qué c quedarnos.

In [352]:
##Siendo estrictos:
def funcion_qmin(D, d_0):
    c = 4
    i = 0
    while i < c:
        d_1 = 999999
        while d_1 > d_0:
            # START QUANTUM PROCESS
            d = []
            for j in D:
                if j <= d_0:
                    d.append(j)
            M = len(d)
            d_1 = np.random.choice(d)
            # END QUANTUM PROCESS
            print(d_1, d)
        if d_1 < d_0:
            i=0
        else:
            i += 1
        d_0 = d_1
        print("iter i:" , i)
    return d_0

D = D_original
d_0 = np.random.choice(D)
d_0= funcion_qmin(D, d_0)
print("Valor:              ", d_0)
    

9 [7, 9, 12, 15, 18]
iter i: 0
9 [7, 9]
iter i: 1
9 [7, 9]
iter i: 2
9 [7, 9]
iter i: 3
7 [7, 9]
iter i: 0
7 [7]
iter i: 1
7 [7]
iter i: 2
7 [7]
iter i: 3
7 [7]
iter i: 4
Valor:               7


In [ ]:
## Mejorando :
def grover_long()

def funcion_qmin(D, d_0):
    d_1 = 999999
    c = 10
    for i in range(c):
        while d_1 > d_0:
            # GROVERLONGO
            d = []
            for i in D:
                if i < d_0:
                    d.append(i)
            M = len(d)
            if len(d) == 0:
                # Esta condición es para evitar que el código
                #  se quede en un bucle adicional si no hay elementos
                #  menores que d_0
                break
            d_1 = np.random.choice(d)
            print(d_1, d)
        if d_1 < d_0:
            d_0 = d_1
    return d_0, M

M=2
D = D_original
d_0 = np.random.choice(D)
while M > 1:
    d_0, M = funcion_qmin(D, d_0)
    print("Valor: ", d_0)

> (1) Each data value is represented by a binary string and is stored in an orthonormal basis state of |Ø>., where |Ø>. is the initial state. Therefore, the data value lies in interval [0,2^n - 1], where n is the number of qubits.<br>
> (2) There is a one-to-one mapping between a data value and its index. The index may be a person's name or other non-numeric data.


En esta sección comienza la codificación cuántica de los datos clásicos $D$ de longitud $N$, siguiendo la estructura de ejemplo del paper, vamos a codificar los datos en 6 qubits superpuestos.

In [4]:
D

[23, 26, 60, 7, 9, 12, 15, 18]

In [6]:
# Codificación binaria de los datos en D usando el número mínimo de bits
n_qubits = int(np.ceil(np.log2(max(D) + 1)))
D_bin = [format(x, f"0{n_qubits}b") for x in D]

print(f"Número de qubits necesarios: {n_qubits}")
for idx, (valor, binario) in enumerate(zip(D, D_bin)):
    print(f"Índice {idx}: {valor} -> |{binario}>")
print(D_bin)

Número de qubits necesarios: 6
Índice 0: 23 -> |010111>
Índice 1: 26 -> |011010>
Índice 2: 60 -> |111100>
Índice 3: 7 -> |000111>
Índice 4: 9 -> |001001>
Índice 5: 12 -> |001100>
Índice 6: 15 -> |001111>
Índice 7: 18 -> |010010>
['010111', '011010', '111100', '000111', '001001', '001100', '001111', '010010']


In [207]:
for i in range(6):
    i = 0
    print("Iteración:", i)

Iteración: 0
Iteración: 0
Iteración: 0
Iteración: 0
Iteración: 0
Iteración: 0
